# GRIT 0.5B on Google Colab (one A100 80GB)

This notebook runs the pinned `Qwen/Qwen2.5-0.5B-Instruct` policy with the pinned `Qwen/Qwen3Guard-Gen-0.6B` safety reward. It keeps the same task corpus, 1,000 projector contexts, 1,920 KL contexts, top-64 plus tail trust region, GRPO grouping, projected AdamW predictor, and central-difference correction as `main.ipynb`.

Choose **Runtime → Change runtime type → A100 80GB GPU**. The actor and persistent vLLM process share GPU 0; this Colab profile is restricted to one actor and the 0.5B policy. Task/evaluation data and preservation prompts are loaded from the cloned repository; missing frozen-base contexts are generated in cell 4. The projector is built once and then reused from Google Drive, alongside complete checkpoints. Colab GPU time and Drive storage are required; the full 40-step duration is unmeasured.

Run cells in order. Context generation, the first projector build, and the training cell may take a long time. Do not put credentials in the notebook or repository.


## 1. Mount Drive and load the repository with prepared data

Set `REPO_URL` to the GitHub repository containing the code and prepared data. The clone must include task/evaluation data and preservation prompt parquets with their manifests. Cell 4 generates missing model-specific contexts from those prompts without resampling them. If using a Drive copy instead, leave `REPO_URL` empty and place the repository including `data/` at `DRIVE_REPO`. A private GitHub repo must be accessible to the Colab runtime without embedding a token in the URL.

If `/content/GRIT` already exists, it is reused. Update that checkout if the required files were pushed after it was cloned. For a repository using Git LFS, retrieve its LFS files before the input check below.


In [1]:
print('[Cell 1/7] Starting...', flush=True)
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, subprocess, sys

def run_with_progress(command, *, env=None):
    # Relay child stdout/stderr through the notebook's Python output stream.
    # text=True converts tqdm carriage returns into readable progress lines.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env={**os.environ, **(env or {}), 'PYTHONUNBUFFERED': '1'}) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


drive.mount('/content/drive')
REPO_URL = 'https://github.com/namdeptraivcd/GRIT.git'  # e.g. https://github.com/your-name/GRIT.git
DRIVE_REPO = Path('/content/drive/MyDrive/GRIT-source')
REPO = Path('/content/GRIT')
if not REPO.exists():
    if REPO_URL:
        run_with_progress(['git', 'clone', '--progress', '--depth', '1', REPO_URL, str(REPO)])
    else:
        assert (DRIVE_REPO / 'scripts/train_grit.py').is_file(), 'Set REPO_URL or copy the repository with data and artifacts to DRIVE_REPO.'
        shutil.copytree(DRIVE_REPO, REPO, ignore=shutil.ignore_patterns('.git', 'checkpoints', 'outputs', '__pycache__'))
assert (REPO / 'grit/model_profiles.py').is_file(), 'Source copy is stale; sync the new small-model support.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source:', REPO)

print('[Cell 1/7] Complete.', flush=True)


Mounted at /content/drive
Cloning into '/content/GRIT'...
remote: Enumerating objects: 126, done.        
remote: Counting objects:   0% (1/126)        
remote: Counting objects:   1% (2/126)        
remote: Counting objects:   2% (3/126)        
remote: Counting objects:   3% (4/126)        
remote: Counting objects:   4% (6/126)        
remote: Counting objects:   5% (7/126)        
remote: Counting objects:   6% (8/126)        
remote: Counting objects:   7% (9/126)        
remote: Counting objects:   8% (11/126)        
remote: Counting objects:   9% (12/126)        
remote: Counting objects:  10% (13/126)        
remote: Counting objects:  11% (14/126)        
remote: Counting objects:  12% (16/126)        
remote: Counting objects:  13% (17/126)        
remote: Counting objects:  14% (18/126)        
remote: Counting objects:  15% (19/126)        
remote: Counting objects:  16% (21/126)        
remote: Counting objects:  17% (22/126)        
remote: Counting objects:  18% (23/126

## 2. Install the same PyTorch/vLLM stack and check the GPU

Use a fresh Colab GPU runtime. This installs the repository's Linux/CUDA dependencies; if Colab asks for a runtime restart, restart and rerun the setup cells before checking the prepared inputs.


In [2]:
print('[Cell 2/7] Starting...', flush=True)
run_with_progress([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-vllm.txt'])
run_with_progress(['nvidia-smi'])
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, 'Select a one-GPU A100 80GB runtime.'
properties = torch.cuda.get_device_properties(0)
assert 'A100' in properties.name and properties.total_memory >= 75_000_000_000, f'Need A100 80GB; got {properties.name}, {properties.total_memory} bytes.'
print(properties.name, round(properties.total_memory / 1e9, 1), 'GB')

print('[Cell 2/7] Complete.', flush=True)


[Cell 2/7] Starting...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

## 3. Configure persistent storage and Hub access

Create `HF_TOKEN` in **Colab Secrets** and grant this notebook access. Give it permission to create/write the private checkpoint repo. If Colab Secrets is unavailable or times out, the next cell falls back to a hidden prompt for this runtime only; the token is not written to the notebook. The repository includes 1,000 held-out prompts sampled deterministically from the pinned PKU-SafeRLHF **test** split. The trainer checks their disjointness from preservation prompts before training; use the existing held-out evaluation split. Training generates and safety-scores one greedy response for every held-out prompt after each 2-step update interval. Input data is read from the repository; generated contexts are saved under `DATA_ROOT`, while projectors and checkpoints are persisted on Drive. Keep the generated contexts when reusing a saved projector; they are validated against its construction corpus. With `save_steps=2`, twenty full checkpoints can require roughly 120 GB just for 0.5B FP32 weights and AdamW state; check your Drive quota before training.


In [3]:
print('[Cell 3/7] Starting...', flush=True)
from grit.model_profiles import SMALL
from scripts.preservation_data import preservation_paths

DRIVE_ROOT = Path('/content/drive/MyDrive/GRIT_colab_0p5b')
DATA_ROOT = REPO / 'data'  # Input prompts from GitHub; generated contexts are saved here.
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'  # Build once, then reuse across Colab sessions.
OUTPUT_ROOT = DRIVE_ROOT / 'checkpoints'
EVALUATION_FILE = REPO / 'data/evaluation/pku_saferlhf_test_1000.parquet'
HUB_REPO_ID = 'fushinguyenex/GRIT'  # e.g. your-name/grit-qwen25-0p5b
RUN_ID = 'qwen25-0p5b-grit-v1'  # Keep fixed across Colab reconnects for resume.

from getpass import getpass

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    try:
        HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip()
    except Exception as exc:
        print(f'Colab Secrets unavailable ({type(exc).__name__}); using hidden input instead.')
if not HF_TOKEN:
    HF_TOKEN = getpass('HF_TOKEN (input hidden): ').strip()
assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets and grant notebook access.'
assert HUB_REPO_ID, 'Set HUB_REPO_ID.'
os.environ['HF_TOKEN'] = HF_TOKEN
for directory in (ARTIFACT_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
paths = preservation_paths(DATA_ROOT, ARTIFACT_ROOT, SMALL.policy)
TASK_FILE = DATA_ROOT / 'task/task_train.parquet'
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
print({'policy': SMALL.policy, 'reward': SMALL.safety, 'data': str(DATA_ROOT), 'run': str(OUTPUT_DIR)})

print('[Cell 3/7] Complete.', flush=True)


[Cell 3/7] Starting...
Colab Secrets unavailable (TimeoutException); using hidden input instead.
{'policy': 'Qwen/Qwen2.5-0.5B-Instruct', 'reward': 'Qwen/Qwen3Guard-Gen-0.6B', 'data': '/content/GRIT/data', 'run': '/content/drive/MyDrive/GRIT_colab_0p5b/checkpoints/qwen25-0p5b-grit-v1'}
[Cell 3/7] Complete.


## 4. Generate frozen-base contexts and build the projector

Use the checked-in prompts: 1,000 projector responses (334 general, 333 math, 333 code) without top-k statistics, then only 1,920 monitor responses (640 per domain, selected from the 6,000 prompt pool). This covers 48 responses × 40 training steps without repetition. Monitor contexts retain frozen-base top-64 plus tail statistics.

Complete local or Drive caches are validated and reused. An incompatible 6,000-response cache is renamed before the reduced monitor corpus is generated; old data and interrupted attempts are retained. Logs show model loading, response generation, top-k token forwards, Drive copy bytes, activation batches and projector eigendecompositions.


In [4]:
print('[Cell 4/7] Starting...', flush=True)
from datetime import datetime, timezone
from grit.progress import Progress
from scripts.prepare_preservation_data import validate_context_artifact
import re, shutil

prep_progress = Progress(prefix='Preparation: ')
model_key = re.sub(r'[^a-zA-Z0-9._-]', '--', SMALL.policy).lower()
PROMPT_ROOT = DATA_ROOT / 'preservation/split_v1/prompts'
DRIVE_CTX_ROOT = DRIVE_ROOT / 'contexts' / 'split_v1' / model_key
paths = preservation_paths(DATA_ROOT, ARTIFACT_ROOT, SMALL.policy)
required_data_inputs = {
    'task dataset': TASK_FILE,
    'held-out evaluation dataset': EVALUATION_FILE,
    'projector prompts': PROMPT_ROOT / 'projector/preserve_prompts.parquet',
    'monitor prompts': PROMPT_ROOT / 'monitor/preserve_prompts.parquet',
}
for name, path in required_data_inputs.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing {name}: {path}')
    prep_progress.log(f'Input ready: {name}')


def keep_existing(path):
    if path.is_symlink() or path.exists():
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        backup = path.with_name(path.name + '.previous.' + stamp)
        path.rename(backup)
        prep_progress.log(f'Preserved existing artifact: {backup}')


for role in prep_progress.track(('projector', 'monitor'), 'Context corpora', unit='corpus'):
    prompts_file = PROMPT_ROOT / role / 'preserve_prompts.parquet'
    repo_ctx_dir = Path(paths['projector_contexts' if role == 'projector' else 'preserve_file']).parent
    drive_ctx_dir = DRIVE_CTX_ROOT / repo_ctx_dir.name
    options = dict(model_path=SMALL.policy, revision=SMALL.policy_revision,
                   limit=1920 if role == 'monitor' else 0, require_statistics=role == 'monitor')
    # Check local and Drive artifacts before spending GPU time. Old model-specific
    # projector caches remain usable even if they include redundant statistics.
    candidates = [drive_ctx_dir, repo_ctx_dir]
    if role == 'projector':
        candidates.append(DRIVE_ROOT / 'contexts/split_v1/projector')
    reusable = None
    for candidate in candidates:
        if not candidate.exists():
            continue
        try:
            with prep_progress.phase(f'Checking {role} cache: {candidate}'):
                validate_context_artifact(candidate, prompts_file, **options)
            reusable = candidate
            break
        except (OSError, ValueError, KeyError) as error:
            prep_progress.log(f'Cache not reusable ({type(error).__name__}): {candidate}')
    if reusable is not None:
        prep_progress.log(f'Reusing {role} contexts: {reusable}')
        if reusable != repo_ctx_dir and repo_ctx_dir.resolve() != reusable.resolve():
            keep_existing(repo_ctx_dir)
            repo_ctx_dir.parent.mkdir(parents=True, exist_ok=True)
            repo_ctx_dir.symlink_to(reusable)
    else:
        keep_existing(repo_ctx_dir)
        count = 1000 if role == 'projector' else 1920
        prep_progress.log(f'Generating {role}: {count} responses')
        role_args = ['--skip-base-statistics'] if role == 'projector' else ['--limit', '1920']
        run_with_progress([
            sys.executable, '-u', 'scripts/prepare_preservation_data.py', 'generate',
            '--prompts', str(prompts_file), '--output-dir', str(repo_ctx_dir),
            '--model-path', SMALL.policy, '--revision', SMALL.policy_revision,
            '--max-prompt-length', '2048', '--max-new-tokens', '256',
            '--top-k', '64', '--dtype', 'float16', '--device', 'cuda', *role_args,
        ])
        validate_context_artifact(repo_ctx_dir, prompts_file, **options)
    # Persist newly generated or previously local contexts. Write manifest last.
    if repo_ctx_dir.resolve() != drive_ctx_dir.resolve():
        keep_existing(drive_ctx_dir)
        drive_ctx_dir.mkdir(parents=True, exist_ok=True)
        files = sorted(repo_ctx_dir.iterdir(), key=lambda path: path.name == 'manifest.json')
        for src in prep_progress.track(files, f'Saving {role} to Drive', unit='file'):
            with src.open('rb') as source, (drive_ctx_dir / src.name).open('wb') as target:
                with prep_progress.bar(src.name, total=src.stat().st_size, unit='B', unit_scale=True) as bar:
                    for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
                        target.write(block)
                        bar.update(len(block))
        validate_context_artifact(drive_ctx_dir, prompts_file, **options)
        prep_progress.log(f'Saved {role} contexts: {drive_ctx_dir}')

# The shell validates both context corpora and reuses a matching projector.
projector_path = Path(paths['projectors_path'])
prep_env = dict(os.environ, DATA_ROOT=str(DATA_ROOT), ARTIFACT_ROOT=str(ARTIFACT_ROOT),
                MODEL_PATH=SMALL.policy, MODEL_REVISION=SMALL.policy_revision,
                PROJECTORS_PATH=str(projector_path), PYTHON_BIN=sys.executable,
                PYTHONPATH=str(REPO))
run_with_progress(['bash', 'scripts/run_preservation_projectors.sh', 'build'], env=prep_env)
assert projector_path.is_file() and projector_path.stat().st_size > 0, projector_path
prep_progress.log(f'Projector ready: {projector_path}')

print('[Cell 4/7] Complete.', flush=True)


[Cell 4/7] Starting...
Preparation: Input ready: task dataset
Preparation: Input ready: held-out evaluation dataset
Preparation: Input ready: projector prompts
Preparation: Input ready: monitor prompts
Preparation: Generating projector: 1000 responses                      
Preparation: Context corpora:   0%|          | 0/2 [00:00<?, ?corpus/s]projector: [1/4] Loading generation libraries...
projector: [2/4] Reading prompts: /content/GRIT/data/preservation/split_v1/prompts/projector/preserve_prompts.parquet

projector: [3/4] Loading frozen base model/tokenizer | elapsed 00:00`torch_dtype` is deprecated! Use `dtype` instead!
2026-10-03 15:41:46.199271: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-03 15:41:46.270288: I tensorflow/core/platform/cpu_feature_guard.cc

## 5. Verify CPU invariants and configure training

The global batch, optimizer, KL support, curvature settings and frozen data match `main.ipynb`. One actor handles all 250 prompts per step. `rollout_memory_utilization=0.2` reserves a smaller GPU share for vLLM because it shares the A100 with the actor; this needs a real GPU fit check.


In [ ]:
# from pathlib import Path

# p = Path('/content/GRIT/test_function/test_training.py')
# s = p.read_text()
# s = s.replace(
#     'from transformers import AutoModelForCausalLM, Qwen2Config, Qwen3Config',
#     'from transformers import Qwen2Config, Qwen2ForCausalLM, Qwen3Config, Qwen3ForCausalLM',
# )
# s = s.replace(
#     'AutoModelForCausalLM.from_config(policy)',
#     'Qwen2ForCausalLM(policy)',
# )
# s = s.replace(
#     'AutoModelForCausalLM.from_config(safety)',
#     'Qwen3ForCausalLM(safety)',
# )
# p.write_text(s)
# print('Đã cập nhật test.')

Đã cập nhật test.


In [10]:
print('[Cell 5/7] Starting...', flush=True)
run_with_progress([sys.executable, '-m', 'pytest', '-v', 'test_function/test_training.py', 'test_function/test_notebook.py'])
config = {
    'task_file': str(TASK_FILE),
    'preserve_file': paths['preserve_file'],
    'evaluation_file': str(EVALUATION_FILE),
    'projectors_path': paths['projectors_path'],
    'output_dir': str(OUTPUT_DIR),
    'model_path': SMALL.policy,
    'model_revision': SMALL.policy_revision,
    'safety_model_path': SMALL.safety,
    'safety_model_revision': SMALL.safety_revision,
    'rollout_gpu': '0',
    'allow_colocated_rollout': True,
    'rollout_memory_utilization': 0.2,
    'rollout_dtype': 'auto',
    'task_optimizer': 'adamw',
    'projector_relaxation': 0.05,
    'task_batch_size': 250,
    'task_microbatch_size': 24,
    'preserve_batch_size': 48,
    'preservation_microbatch_size': 2,
    'generations': 5,
    'max_prompt_length': 512,
    'max_response_length': 256,
    'max_preserve_length': 2304,
    'lr': 1e-6,
    'lambda_pres': 1.0,
    'epsilon_pres': 0.05,
    'top_k': 64,
    'base_statistics_dtype': 'float16',
    'use_curvature': True,
    'fd_radius': 0.05,
    'fd_check_interval': 0,
    'gradient_topk': 5,
    'gradient_checkpointing': True,
    'reward_batch_size': 32,
    'max_steps': 40,
    'progress': True,
    'save_steps': 5,
    'validation_steps': 5,
    'hub_repo_id': HUB_REPO_ID,
    'seed': 66,
}
from scripts.train_grit import parse_args
arguments = []
for key, value in config.items():
    flag = '--' + key.replace('_', '-')
    if isinstance(value, bool):
        if value:
            arguments.append(flag)
    elif value is not None:
        arguments.extend([flag, str(value)])
parse_args(arguments)
print('Training configuration validated; output:', OUTPUT_DIR)

print('[Cell 5/7] Complete.', flush=True)


[Cell 5/7] Starting...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/GRIT
configfile: pytest.ini
plugins: anyio-4.15.1, langsmith-0.12.5, typeguard-4.6.0
collecting ... collected 40 items

test_function/test_training.py::test_full_correction_matches_unrolled_autograd[0.0-0.005-sgd] PASSED [  2%]
test_function/test_training.py::test_full_correction_matches_unrolled_autograd[0.0-0.005-adamw] PASSED [  5%]
test_function/test_training.py::test_full_correction_matches_unrolled_autograd[0.0-0.01-sgd] PASSED [  7%]
test_function/test_training.py::test_full_correction_matches_unrolled_autograd[0.0-0.01-adamw] PASSED [ 10%]
test_function/test_training.py::test_full_correction_matches_unrolled_autograd[0.05-0.005-sgd] PASSED [ 12%]
test_function/test_training.py::test_full_correction_matches_unrolled_autograd[0.05-0.005-adamw] PASSED [ 15%]

usage: colab_kernel_launcher.py [-h] [--model-path MODEL_PATH]
                                [--model-revision MODEL_REVISION]
                                --task-file TASK_FILE
                                [--preserve-file PRESERVE_FILE]
                                [--evaluation-file EVALUATION_FILE]
                                --projectors-path PROJECTORS_PATH
                                [--projector-relaxation PROJECTOR_RELAXATION]
                                [--output-dir OUTPUT_DIR] [--resume RESUME]
                                --rollout-gpu ROLLOUT_GPU
                                [--allow-colocated-rollout]
                                [--rollout-dtype {auto,float32,bfloat16,float16}]
                                [--rollout-memory-utilization ROLLOUT_MEMORY_UTILIZATION]
                                [--rollout-timeout ROLLOUT_TIMEOUT]
                                [--task-batch-size TASK_BATCH_SIZE]
                                [--pres

SystemExit: 2

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 6. Train or resume from the latest complete Drive checkpoint

This cell starts the GPU training job. Checkpoints are saved every five steps. If Colab ends, reconnect, rerun the setup cells, and execute this cell again; it selects the latest checkpoint with `complete.json`. Keep `RUN_ID` and config unchanged. An interrupted checkpoint directory is renamed and retained so the next save can use its step number. If a run folder exists without any complete checkpoint, inspect it and use a new `RUN_ID`; no data is deleted automatically.


In [9]:
print('[Cell 6/7] Starting...', flush=True)
from datetime import datetime, timezone
if OUTPUT_DIR.exists():
    for candidate in sorted(OUTPUT_DIR.glob('step_*')):
        if candidate.is_dir() and not (candidate / 'complete.json').is_file():
            stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
            archived = OUTPUT_DIR / f'interrupted_{candidate.name}_{stamp}'
            candidate.rename(archived)
            print('Preserved interrupted checkpoint:', archived)
completed = sorted(OUTPUT_DIR.glob('step_*/complete.json'),
                   key=lambda path: int(path.parent.name.split('_')[-1])) if OUTPUT_DIR.exists() else []
train_args = arguments.copy()
if completed:
    last = completed[-1].parent
    last_step = int(last.name.split('_')[-1])
    if last_step >= config['max_steps']:
        print('Training already complete at step', last_step)
    else:
        train_args.extend(['--resume', str(last)])
        print('Resuming from', last)
elif OUTPUT_DIR.exists():
    raise RuntimeError(f'Incomplete run folder without a complete checkpoint: {OUTPUT_DIR}')
if not completed or int(completed[-1].parent.name.split('_')[-1]) < config['max_steps']:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', PYTHONPATH=str(REPO), OMP_NUM_THREADS='4')
    run_with_progress([sys.executable, '-u', 'scripts/train_grit.py', *train_args], env=env)

print('[Cell 6/7] Complete.', flush=True)


[Cell 6/7] Starting...


NameError: name 'arguments' is not defined

## 7. Inspect persisted results

`metrics.jsonl`, `diagnostics_summary.json`, complete checkpoints and optimizer state stay in Drive. The first step records KL at the frozen base against the stored top-64 statistics.


In [ ]:
print('[Cell 7/7] Starting...', flush=True)
import json
metrics_path = OUTPUT_DIR / 'metrics.jsonl'
if metrics_path.is_file():
    metrics = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print('Completed steps:', len(metrics))
    print(json.dumps(metrics[-1], indent=2) if metrics else 'No completed step yet')
summary_path = OUTPUT_DIR / 'diagnostics_summary.json'
if summary_path.is_file():
    print('Diagnostics:', summary_path.read_text())

print('[Cell 7/7] Complete.', flush=True)
